In [11]:
import wikipedia as wiki

page = "Ice Nine Kills"
results = wiki.search(page)
p = wiki.page(results[0])

# derive public attrs from dir(), excluding dunders and private
public_attrs = [a for a in dir(p) if not a.startswith('_')]

for attr in public_attrs:
    print(f"\n{'='*60}")
    print(f"ATTRIBUTE: {attr}")
    print(f"{'='*60}")
    try:
        val = getattr(p, attr)
        # skip callables except section() which we handle separately
        if callable(val):
            print(f"<callable>")
            continue
        print(f"TYPE: {type(val).__name__}")
        if isinstance(val, str):
            print(val[:500] + ("..." if len(val) > 500 else ""))
        elif isinstance(val, list):
            print(f"Length: {len(val)}")
            print(val[:10])
        else:
            print(val)
    except Exception as e:
        print(f"ERROR: {e}")

# handle section() separately since it needs an argument
print(f"\n{'='*60}")
print("METHOD: section() — iterating all sections")
print(f"{'='*60}")
try:
    for s in p.sections:
        print(f"\n--- {s} ---")
        text = p.section(s)
        if text:
            print(text[:300] + ("..." if len(text) > 300 else ""))
        else:
            print("<empty>")
except Exception as e:
    print(f"ERROR: {e}")


ATTRIBUTE: categories
TYPE: list
Length: 20
['2000 establishments in Massachusetts', 'All Wikipedia articles written in American English', 'All articles lacking reliable references', 'American musical quintets', 'Articles lacking reliable references from August 2014', 'Articles with hCards', 'Articles with short description', 'CS1 German-language sources (de)', 'Commons category link from Wikidata', 'Fearless Records artists']

ATTRIBUTE: content
TYPE: str
Ice Nine Kills is an American heavy metal band from Boston, Massachusetts. The band's music is primarily described as horror-themed metalcore, but incorporates many other musical styles. It was originally a ska punk band called Ice Nine, formed in 2000 by high school friends Spencer Charnas and Jeremy Schwartz, before adopting a metalcore style in 2010. Charnas is currently the only remaining founding member. The band is signed to Fearless Records.
Ice Nine Kills has released three EPs along wit...

ATTRIBUTE: coordinates
ERROR: 'co

In [13]:
import requests
import re

HEADERS = {"User-Agent": "msds565-course-demo/1.0 (contact@example.com)"}

def get_sections(title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "parse",
        "page": title,
        "prop": "sections",
        "format": "json"
    }
    response = requests.get(url, params=params, headers=HEADERS)
    print(f"Status: {response.status_code}")
    print(f"Raw (first 300 chars): {response.text[:300]}")
    return response.json()["parse"]["sections"]

def get_section_text(title, section_index):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "content",
        "rvsection": section_index,
        "format": "json"
    }
    response = requests.get(url, params=params, headers=HEADERS)
    pages = response.json()["query"]["pages"]
    page = next(iter(pages.values()))
    return page["revisions"][0]["*"]

def clean_wikitext(raw):
    text = re.sub(r'\{\{[^}]*\}\}', '', raw)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r"'{2,}", '', text)
    text = re.sub(r'==+[^=]+=+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\n{2,}', '\n', text)
    return text.strip()

# --- demo ---
title = "Ice Nine Kills"

sections = get_sections(title)
print(f"\nFound {len(sections)} sections:\n")
for s in sections:
    print(f"  [{s['index']}] {'  ' * (s['toclevel']-1)}{s['line']}")

print("\n" + "="*60)
print("SECTION TEXT DEMO")
print("="*60)

for s in sections[:4]:
    idx = s["index"]
    heading = s["line"]
    raw = get_section_text(title, idx)
    clean = clean_wikitext(raw)
    print(f"\n--- {heading} (index {idx}) ---")
    print(f"RAW (first 200 chars):\n{raw[:200]}")
    print(f"\nCLEANED (first 200 chars):\n{clean[:200]}")
    print()

Status: 200
Raw (first 300 chars): {"warnings":{"main":{"*":"Subscribe to the mediawiki-api-announce mailing list at <https://lists.wikimedia.org/postorius/lists/mediawiki-api-announce.lists.wikimedia.org/> for notice of API deprecations and breaking changes. Use [[Special:ApiFeatureUsage]] to see usage of deprecated features by your

Found 12 sections:

  [1] History
  [2]   2000–2009: Formation and <i>Last Chance to Make Amends</i>
  [3]   2010–2013: <i>Safe Is Just a Shadow</i> and <i>The Predator</i>
  [4]   2014–2015: <i>The Predator Becomes the Prey</i>
  [5]   2015–2018: <i>Every Trick in the Book</i>
  [6]   2018–present: <i>The Silver Scream</i> series
  [7] Musical style
  [8] Band members
  [9] Discography
  [10] Accolades
  [11] References
  [12] External links

SECTION TEXT DEMO

--- History (index 1) ---
RAW (first 200 chars):
== History ==

=== 2000–2009: Formation and ''Last Chance to Make Amends'' ===
Ice Nine Kills was founded in 2000 under the name Ice Nine by high s